<a href="https://colab.research.google.com/github/Manjot-Coder/IP_LabAssignments/blob/main/Assignmen-7/All%20questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Q1,Q2,Q3


from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io

# Step 1: Upload image
uploaded = files.upload()
filename = next(iter(uploaded))

img = Image.open(io.BytesIO(uploaded[filename])).convert("L")
image = np.array(img, dtype=float)


# Function to apply convolution
def apply_filter(image, kernel):
    k = kernel.shape[0]
    pad = k // 2

    padded = np.pad(image, pad, mode="edge")
    result = np.zeros(image.shape, dtype=float)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i:i+k, j:j+k]
            result[i, j] = np.sum(region * kernel)

    return result


# Function to convert result to displayable image
def normalize(image):
    image = np.abs(image)
    if image.max() == 0:
        return image.astype(np.uint8)
    return (image / image.max() * 255).astype(np.uint8)


# ------------------------------------------------
# 1. LAPLACIAN FILTER
# ------------------------------------------------

laplacian_kernel = np.array([
    [0,  1, 0],
    [1, -4, 1],
    [0,  1, 0]
])

laplacian = apply_filter(image, laplacian_kernel)

# Sharpening: Original - Laplacian
laplacian_sharp = np.clip(image - laplacian, 0, 255)

laplacian_display = normalize(laplacian)


# ------------------------------------------------
# 2. SOBEL EDGE DETECTION
# ------------------------------------------------

sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

sobel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
])

gx = apply_filter(image, sobel_x)
gy = apply_filter(image, sobel_y)

sobel_magnitude = np.sqrt(gx**2 + gy**2)
sobel_edges = normalize(sobel_magnitude)

# Sharpen using Sobel edges
sobel_sharp = np.clip(image + sobel_magnitude, 0, 255)


# ------------------------------------------------
# 3. CANNY EDGE DETECTOR (MANUAL)
# ------------------------------------------------

# Step 1: Gaussian smoothing (5x5, sigma=1)
sigma_canny = 1
size = 5
center = size // 2

gaussian = np.zeros((size, size))

for i in range(size):
    for j in range(size):
        x = i - center
        y = j - center
        gaussian[i, j] = np.exp(
            -(x*x + y*y) / (2*sigma_canny**2)
        )

gaussian /= gaussian.sum()

smooth = apply_filter(image, gaussian)

# Step 2: Sobel gradients
gx = apply_filter(smooth, sobel_x)
gy = apply_filter(smooth, sobel_y)

magnitude = np.sqrt(gx**2 + gy**2)
angle = np.arctan2(gy, gx) * 180 / np.pi
angle = (angle + 180) % 180

# Step 3: Non-maximum suppression
nms = np.zeros(magnitude.shape)
padded_mag = np.pad(magnitude, 1, mode="edge")

for i in range(1, magnitude.shape[0] + 1):
    for j in range(1, magnitude.shape[1] + 1):

        a = angle[i-1, j-1]
        current = padded_mag[i, j]

        if (0 <= a < 22.5) or (157.5 <= a <= 180):
            n1, n2 = padded_mag[i, j-1], padded_mag[i, j+1]

        elif 22.5 <= a < 67.5:
            n1, n2 = padded_mag[i-1, j+1], padded_mag[i+1, j-1]

        elif 67.5 <= a < 112.5:
            n1, n2 = padded_mag[i-1, j], padded_mag[i+1, j]

        else:
            n1, n2 = padded_mag[i-1, j-1], padded_mag[i+1, j+1]

        if current >= n1 and current >= n2:
            nms[i-1, j-1] = current

# Step 4: Double threshold
high = nms.max() * 0.20
low = high * 0.50

strong = 255
weak = 75

edges = np.zeros(nms.shape, dtype=np.uint8)

edges[nms >= high] = strong
edges[(nms >= low) & (nms < high)] = weak

# Step 5: Edge tracking by hysteresis
for i in range(1, edges.shape[0]-1):
    for j in range(1, edges.shape[1]-1):

        if edges[i, j] == weak:
            if np.any(edges[i-1:i+2, j-1:j+2] == strong):
                edges[i, j] = strong
            else:
                edges[i, j] = 0

canny_edges = edges

# Sharpen using Canny edges
canny_sharp = np.clip(
    image + (canny_edges.astype(float) / 255) * 100,
    0, 255
)


# ------------------------------------------------
# 4. UNSHARP MASKING
# Gaussian filter size = 5x5, sigma = 3
# ------------------------------------------------

size = 5
sigma = 3
center = size // 2

gaussian = np.zeros((size, size))

for i in range(size):
    for j in range(size):
        x = i - center
        y = j - center

        gaussian[i, j] = np.exp(
            -(x*x + y*y) / (2*sigma*sigma)
        )

gaussian /= gaussian.sum()

# Blur image
blurred = apply_filter(image, gaussian)

# Create mask
mask = image - blurred

# Sharpened image
unsharp = np.clip(image + mask, 0, 255)


# ------------------------------------------------
# DISPLAY ALL RESULTS
# ------------------------------------------------

plt.figure(figsize=(14, 10))

plt.subplot(3, 3, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Image")
plt.axis("off")

plt.subplot(3, 3, 2)
plt.imshow(laplacian_display, cmap="gray")
plt.title("Laplacian Edges")
plt.axis("off")

plt.subplot(3, 3, 3)
plt.imshow(laplacian_sharp, cmap="gray")
plt.title("Laplacian Sharpened")
plt.axis("off")

plt.subplot(3, 3, 4)
plt.imshow(sobel_edges, cmap="gray")
plt.title("Sobel Edges")
plt.axis("off")

plt.subplot(3, 3, 5)
plt.imshow(sobel_sharp, cmap="gray")
plt.title("Sobel Sharpened")
plt.axis("off")

plt.subplot(3, 3, 6)
plt.imshow(canny_edges, cmap="gray")
plt.title("Canny Edges")
plt.axis("off")

plt.subplot(3, 3, 7)
plt.imshow(canny_sharp, cmap="gray")
plt.title("Canny Sharpened")
plt.axis("off")

plt.subplot(3, 3, 8)
plt.imshow(blurred, cmap="gray")
plt.title("Gaussian Blurred")
plt.axis("off")

plt.subplot(3, 3, 9)
plt.imshow(unsharp, cmap="gray")
plt.title("Unsharp Masking")
plt.axis("off")

plt.tight_layout()
plt.show()


# Save output images
Image.fromarray(laplacian_sharp.astype(np.uint8)).save("laplacian_sharp.jpg")
Image.fromarray(sobel_sharp.astype(np.uint8)).save("sobel_sharp.jpg")
Image.fromarray(canny_sharp.astype(np.uint8)).save("canny_sharp.jpg")
Image.fromarray(unsharp.astype(np.uint8)).save("unsharp_masking.jpg")

print("All sharpened images saved successfully!")
